# 06 - XGBoost (tabular branch)

Gradient boosting on the heterogeneous, missing-value-heavy certificate and DNS features - where trees still beat deep learning, and where SHAP's TreeExplainer gives exact Shapley values.

In [ ]:
# --- standard header: every notebook starts with exactly this ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'

if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)
else:
    # Private repo: paste your GitHub Personal Access Token when prompted.
    # It is only held in this runtime and vanishes when the session ends.
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{URL}', REPO], check=True)

sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))


In [ ]:
import pandas as pd
from src.models import xgb as xgbm
from src.evaluate import splits, metrics, predictions
from src.features.build import to_matrix

cfg = config.load('xgboost')
df = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet")
split = splits.load_split(P['data']['splits'], cfg['split']['name'])
tr, va, te = splits.apply_split(df, split)
Xtr, ytr, names = to_matrix(tr); Xva, yva, _ = to_matrix(va); Xte, yte, _ = to_matrix(te)

In [ ]:
model = xgbm.build(cfg['model'], ytr)
model = xgbm.fit(model, Xtr, ytr, Xva, yva)
scores = model.predict_proba(Xte)[:, 1]
m = metrics.evaluate(yte, scores)
m

In [ ]:
counter = manifest.next_counter(P['manifest'])
run_id = manifest.make_run_id('xgb', cfg['split']['name'], cfg['seed'], counter)
model.save_model(f"{P['artifacts']['models']}/{run_id}.json")
predictions.save(run_id, P['artifacts']['predictions'], te['domain'], yte, scores)
manifest.record(P['manifest'], run_id, 'xgboost', cfg, cfg['split']['name'],
                split['split_file'], m, cfg['seed'], repo_root=REPO)
print(run_id)